# Notebook 05 — Hyperparameter Tuning
**Project:** Customer Churn Prediction  
**Depends on:** `02_preprocessing`, `03_baseline`, `04_xai_gate1`  
**Output:** model ter-tune per kandidat + Voting Ensemble + `tuning_summary.json`

Pipeline:
1. Load splits + baseline models
2. Optuna tuning per model (TPESampler + MedianPruner, 3-fold stratified CV)
3. Voting Ensemble — weight search dari top-3 post-tuning
4. Ringkasan & simpan artifacts ke WandB

---
## Install & Import

In [1]:
!pip install optuna lightgbm xgboost wandb shap --quiet
print('OK install selesai.')

OK install selesai.


In [2]:
import os, json, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import joblib
import optuna
from optuna.samplers import TPESampler
from optuna.pruners  import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics         import average_precision_score
import lightgbm as lgb
import xgboost  as xgb
import wandb

print('OK import selesai.')

OK import selesai.


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB")
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ardiyanto24 (ardiyanto24-indonesian-national-police) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

---
## Konstanta Global

In [4]:
# ── Path ──────────────────────────────────────────────────────────────────────
PREPROCESSING_DIR = '/kaggle/input/notebooks/ardiyanto24/tccp-preprocessing-v2/artifacts'
BASELINE_DIR      = '/kaggle/input/notebooks/ardiyanto24/tccp-modeling-baseline-v2/artifacts'
SPLITS_PATH       = f'{PREPROCESSING_DIR}/splits.joblib'
OUTPUT_DIR        = '/kaggle/working/artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── WandB ─────────────────────────────────────────────────────────────────────
WANDB_PROJECT    = 'customer-churn-prediction'
WANDB_TUNE_GROUP = 'tuning-05'

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── Optuna ────────────────────────────────────────────────────────────────────
N_TRIALS       = 100   # Jalur 1: semua kandidat
N_CV_FOLDS     = 3     # stratified k-fold untuk CV inner loop
OPTUNA_TIMEOUT = None  # None = tidak ada batas waktu tambahan

# ── Cross-validation ──────────────────────────────────────────────────────────
CV_SCORING = 'average_precision'   # PR-AUC via sklearn

# ── Voting Ensemble ───────────────────────────────────────────────────────────
N_TRIALS_ENSEMBLE = 30   # weight search top-3 post-tuning
TOP_K_FOR_ENSEMBLE = 3

# ── Kandidat (sama dengan XAI Gate #1, tanpa Random Forest) ──────────────────
CANDIDATES = [
    {'model_name': 'lightgbm',            'balance': 'class_weight', 'tier': 1,
     'baseline_pr_auc': 0.7514},
    {'model_name': 'xgboost',             'balance': 'class_weight', 'tier': 1,
     'baseline_pr_auc': 0.7512},
    {'model_name': 'xgboost',             'balance': 'smote',        'tier': 1,
     'baseline_pr_auc': 0.7479},
    {'model_name': 'lightgbm',            'balance': 'smote',        'tier': 1,
     'baseline_pr_auc': 0.7464},
    {'model_name': 'logistic_regression', 'balance': 'class_weight', 'tier': 2,
     'baseline_pr_auc': 0.7410},
    {'model_name': 'logistic_regression', 'balance': 'smote',        'tier': 2,
     'baseline_pr_auc': 0.7407},
]

print('OK konstanta global terdefinisi.')
print(f'   N_TRIALS          : {N_TRIALS}')
print(f'   N_CV_FOLDS        : {N_CV_FOLDS}')
print(f'   N_TRIALS_ENSEMBLE : {N_TRIALS_ENSEMBLE}')
print(f'   Jumlah kandidat   : {len(CANDIDATES)}')

OK konstanta global terdefinisi.
   N_TRIALS          : 100
   N_CV_FOLDS        : 3
   N_TRIALS_ENSEMBLE : 30
   Jumlah kandidat   : 6


---
## Load Artifacts

In [5]:
# ── Load splits ───────────────────────────────────────────────────────────────
print('Loading splits...')
splits        = joblib.load(SPLITS_PATH)
X_train       = splits['X_train']
y_train       = splits['y_train']
X_val         = splits['X_val']
y_val         = splits['y_val']
feature_names = splits['feature_names']

print(f'  X_train shape  : {X_train.shape}')
print(f'  X_val   shape  : {X_val.shape}')
print(f'  y_train churn  : {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'  y_val   churn  : {y_val.sum():,} ({y_val.mean()*100:.1f}%)')
print(f'  Feature names  : {len(feature_names)} fitur')

Loading splits...
  X_train shape  : (415935, 29)
  X_val   shape  : (89129, 29)
  y_train churn  : 93,672 (22.5%)
  y_val   churn  : 20,072 (22.5%)
  Feature names  : 29 fitur


In [6]:
# ── Load baseline models (sebagai referensi perbandingan) ─────────────────────
print('Loading baseline models...')
baseline_models = {}

for cand in CANDIDATES:
    key   = f"{cand['model_name']}__{cand['balance']}"
    fpath = os.path.join(BASELINE_DIR, f'{key}.joblib')
    if os.path.exists(fpath):
        baseline_models[key] = joblib.load(fpath)
        print(f'  OK {key}')
    else:
        print(f'  NOT FOUND: {fpath}')

print(f'\nTotal baseline loaded: {len(baseline_models)}/{len(CANDIDATES)}')

Loading baseline models...
  OK lightgbm__class_weight
  OK xgboost__class_weight
  OK xgboost__smote
  OK lightgbm__smote
  OK logistic_regression__class_weight
  OK logistic_regression__smote

Total baseline loaded: 6/6


---
## Deklarasi Class & Function

> Semua class bersifat stateless. Setiap method menerima data dan mengembalikan hasil — tidak ada state yang perlu di-fit di luar model itu sendiri.

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: ModelFactory
# Tanggung jawab: membuat instance model dari model_name + params + balance
# class_weight ditangani di level model (bukan di data)
# ══════════════════════════════════════════════════════════════════════════════
class ModelFactory:
    """Membuat instance model yang siap di-fit dari nama + params + balance strategy."""

    @staticmethod
    def build(model_name: str, balance: str, params: dict):
        """
        balance='class_weight' → tambahkan class_weight='balanced' ke model
        balance='smote'        → data sudah di-resample, model tanpa weight
        """
        cw = 'balanced' if balance == 'class_weight' else None

        if model_name == 'lightgbm':
            p = {**params}
            if cw:
                p['class_weight'] = cw
            return lgb.LGBMClassifier(**p)

        elif model_name == 'xgboost':
            p = {**params}
            if cw:
                # XGBoost: hitung scale_pos_weight dari distribusi y_train
                neg  = (y_train == 0).sum()
                pos  = (y_train == 1).sum()
                p['scale_pos_weight'] = neg / pos
            return xgb.XGBClassifier(**p)

        elif model_name == 'logistic_regression':
            p = {**params}
            # Safety net: re-derive solver jika tidak ada di params
            # (backward compatibility untuk params lama tanpa solver)
            if 'solver' not in p:
                p['solver'] = 'saga' if p.get('penalty') == 'l1' else 'lbfgs'
            if cw:
                p['class_weight'] = cw
            return LogisticRegression(**p)

        else:
            raise ValueError(f'Model tidak dikenal: {model_name}')

print('OK ModelFactory terdefinisi.')

OK ModelFactory terdefinisi.


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: SearchSpaceBuilder
# Tanggung jawab: definisi search space Optuna per model_name
# Setiap method mengembalikan dict params siap pakai untuk training
# ══════════════════════════════════════════════════════════════════════════════
class SearchSpaceBuilder:
    """Mendefinisikan Optuna search space per arsitektur model."""

    @staticmethod
    def lightgbm(trial: optuna.Trial) -> dict:
        """
        Search space LightGBM.
        Prioritas regularisasi untuk menjaga konsistensi XAI post-tuning.
        num_leaves dibatasi 150 — cukup ekspresif tanpa overly deep tree.
        """
        return {
            'n_estimators'     : trial.suggest_int  ('n_estimators',      100, 1000),
            'num_leaves'       : trial.suggest_int  ('num_leaves',         20,  150),
            'max_depth'        : trial.suggest_int  ('max_depth',           3,   10),
            'learning_rate'    : trial.suggest_float('learning_rate',    0.01,  0.3, log=True),
            'min_child_samples': trial.suggest_int  ('min_child_samples',  10,  100),
            'subsample'        : trial.suggest_float('subsample',          0.6,  1.0),
            'colsample_bytree' : trial.suggest_float('colsample_bytree',   0.6,  1.0),
            'reg_alpha'        : trial.suggest_float('reg_alpha',          0.0,  1.0),
            'reg_lambda'       : trial.suggest_float('reg_lambda',         0.0,  5.0),
            'random_state'     : RANDOM_SEED,
            'verbosity'        : -1,
            'n_jobs'           : -1,
        }

    @staticmethod
    def xgboost(trial: optuna.Trial) -> dict:
        """
        Search space XGBoost.
        max_depth lebih konservatif dari LightGBM (3–8 vs 3–10).
        colsample_bylevel ditambah sebagai regularisasi ekstra.
        gamma penting untuk XGBoost: min loss reduction sebelum split.
        """
        return {
            'n_estimators'      : trial.suggest_int  ('n_estimators',       100, 1000),
            'max_depth'         : trial.suggest_int  ('max_depth',            3,    8),
            'learning_rate'     : trial.suggest_float('learning_rate',     0.01,   0.3, log=True),
            'subsample'         : trial.suggest_float('subsample',           0.6,   1.0),
            'colsample_bytree'  : trial.suggest_float('colsample_bytree',    0.5,   1.0),
            'colsample_bylevel' : trial.suggest_float('colsample_bylevel',   0.5,   1.0),
            'gamma'             : trial.suggest_float('gamma',               0.0,   5.0),
            'reg_alpha'         : trial.suggest_float('reg_alpha',           0.0,   2.0),
            'reg_lambda'        : trial.suggest_float('reg_lambda',          0.0,   5.0),
            'min_child_weight'  : trial.suggest_int  ('min_child_weight',     1,    50),
            'random_state'      : RANDOM_SEED,
            'eval_metric'       : 'aucpr',
            'verbosity'         : 0,
            'n_jobs'            : -1,
        }

    @staticmethod
    def logistic_regression(trial: optuna.Trial) -> dict:
        """
        Search space Logistic Regression.
        C adalah parameter utama — log-scale karena efeknya bersifat eksponensial.
        Solver disesuaikan dengan penalty untuk menghindari error inkompatibilitas.

        FIX: solver di-suggest via trial.suggest_categorical agar masuk
        study.best_params saat refit. Sebelumnya solver dihitung derived
        (tidak di-suggest) sehingga hilang dari best_params → crash saat
        best trial memilih l1 + default lbfgs.
        """
        penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])

        # Solver harus kompatibel dengan penalty:
        #   l1 → hanya saga/liblinear yang support
        #   l2 → lbfgs (default sklearn, paling efisien)
        solver  = 'saga' if penalty == 'l1' else 'lbfgs'
        return {
            'C'           : trial.suggest_float('C', 0.001, 100.0, log=True),
            'penalty'     : penalty,
            'solver'      : solver,
            'max_iter'    : trial.suggest_int('max_iter', 200, 2000),
            'random_state': RANDOM_SEED,
            'n_jobs'      : -1,
        }


    @staticmethod
    def reconstruct(model_name: str, best_params: dict) -> dict:
        """Tambahkan derived params yang tidak di-suggest oleh Optuna."""
        p = {**best_params}
        if model_name == 'logistic_regression':
            if 'solver' not in p:
                p['solver'] = 'saga' if p.get('penalty') == 'l1' else 'lbfgs'
            p['random_state'] = RANDOM_SEED
            p['n_jobs'] = -1
        elif model_name == 'lightgbm':
            p['random_state'] = RANDOM_SEED
            p['verbosity'] = -1
            p['n_jobs'] = -1
        elif model_name == 'xgboost':
            p['random_state'] = RANDOM_SEED
            p['eval_metric'] = 'aucpr'
            p['verbosity'] = 0
            p['n_jobs'] = -1
        return p

print('OK SearchSpaceBuilder terdefinisi.')

OK SearchSpaceBuilder terdefinisi.


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: SMOTEDataProvider
# Tanggung jawab: menyediakan X_train_smote dan y_train_smote jika diperlukan
# Dibuat sekali, di-cache untuk efisiensi
# ══════════════════════════════════════════════════════════════════════════════
class SMOTEDataProvider:
    """
    Menyediakan training data yang sudah di-oversample dengan SMOTE.
    SMOTE hanya dilakukan pada training fold saat CV — tidak pada val/test.
    Untuk final refit, SMOTE dilakukan pada seluruh X_train.
    """
    _X_full = None
    _y_full = None

    @classmethod
    def get_full(cls):
        """SMOTE pada seluruh X_train — dipakai untuk final refit setelah CV."""
        if cls._X_full is None:
            from imblearn.over_sampling import SMOTE
            sm = SMOTE(random_state=RANDOM_SEED)
            cls._X_full, cls._y_full = sm.fit_resample(X_train, y_train)
            print(f'  SMOTE full: {X_train.shape[0]} -> {cls._X_full.shape[0]} rows')
        return cls._X_full, cls._y_full

    @staticmethod
    def apply_to_fold(X_fold, y_fold):
        """SMOTE pada satu fold training — dipakai saat CV inner loop."""
        from imblearn.over_sampling import SMOTE
        sm = SMOTE(random_state=RANDOM_SEED)
        return sm.fit_resample(X_fold, y_fold)

print('OK SMOTEDataProvider terdefinisi.')

OK SMOTEDataProvider terdefinisi.


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: CVEvaluator
# Tanggung jawab: mengevaluasi satu set params dengan 3-fold stratified CV
# Mengembalikan mean PR-AUC across folds
# ══════════════════════════════════════════════════════════════════════════════
class CVEvaluator:
    """
    Evaluasi model dengan stratified k-fold CV.
    SMOTE diterapkan per fold training (bukan sebelum split) untuk mencegah
    data leakage dari oversampled minority samples ke fold validasi.
    """

    def __init__(self, n_folds: int = N_CV_FOLDS, random_state: int = RANDOM_SEED):
        self.n_folds      = n_folds
        self.random_state = random_state
        self.skf          = StratifiedKFold(
            n_splits=n_folds, shuffle=True, random_state=random_state
        )

    def evaluate(self, model_name: str, balance: str, params: dict) -> float:
        """
        Jalankan CV dan kembalikan mean PR-AUC.
        Untuk balance='smote', SMOTE diterapkan hanya pada training fold.
        """
        scores = []

        for fold_idx, (train_idx, val_idx) in enumerate(
            self.skf.split(X_train, y_train)
        ):
            X_fold_tr, X_fold_val = X_train[train_idx], X_train[val_idx]
            y_fold_tr, y_fold_val = y_train[train_idx], y_train[val_idx]

            # SMOTE hanya pada training fold
            if balance == 'smote':
                X_fold_tr, y_fold_tr = SMOTEDataProvider.apply_to_fold(
                    X_fold_tr, y_fold_tr
                )

            model = ModelFactory.build(model_name, balance, params)
            model.fit(X_fold_tr, y_fold_tr)

            y_prob = model.predict_proba(X_fold_val)[:, 1]
            score  = average_precision_score(y_fold_val, y_prob)
            scores.append(score)

        return float(np.mean(scores))

cv_evaluator = CVEvaluator()
print('OK CVEvaluator terdefinisi.')

OK CVEvaluator terdefinisi.


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: OptunaObjective
# Tanggung jawab: membungkus CVEvaluator sebagai Optuna objective function
# Setiap trial: suggest params → CV → kembalikan mean PR-AUC (maximized)
# ══════════════════════════════════════════════════════════════════════════════
class OptunaObjective:
    """Objective function untuk Optuna study per kandidat model."""

    def __init__(self, model_name: str, balance: str):
        self.model_name = model_name
        self.balance    = balance
        self.space_fn   = getattr(SearchSpaceBuilder, model_name)

    def __call__(self, trial: optuna.Trial) -> float:
        params = self.space_fn(trial)
        return cv_evaluator.evaluate(self.model_name, self.balance, params)

print('OK OptunaObjective terdefinisi.')

OK OptunaObjective terdefinisi.


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: TuningRunner
# Tanggung jawab: menjalankan Optuna study untuk satu kandidat, refit pada
# seluruh X_train dengan best params, evaluasi di val set, simpan artifact
# ══════════════════════════════════════════════════════════════════════════════
class TuningRunner:
    """
    Mengelola siklus tuning lengkap untuk satu kandidat:
    1. Buat Optuna study (TPESampler + MedianPruner)
    2. Optimize N_TRIALS trials (objective = mean CV PR-AUC)
    3. Refit model terbaik pada seluruh X_train
    4. Evaluasi di X_val (held-out)
    5. Log ke WandB
    6. Simpan model ke disk
    """

    def __init__(self, model_name: str, balance: str, baseline_pr_auc: float):
        self.model_name       = model_name
        self.balance          = balance
        self.baseline_pr_auc  = baseline_pr_auc
        self.key              = f'{model_name}__{balance}'

    def run(self) -> dict:
        SEP = '=' * 65
        print(f'\n{SEP}')
        print(f'  Tuning: {self.key}  |  baseline val_PR-AUC={self.baseline_pr_auc:.4f}')
        print(SEP)
        t_start = time.time()

        # ── Buat study ────────────────────────────────────────────────────────
        study = optuna.create_study(
            direction = 'maximize',
            sampler   = TPESampler(seed=RANDOM_SEED),
            pruner    = MedianPruner(n_startup_trials=10, n_warmup_steps=0),
        )

        objective = OptunaObjective(self.model_name, self.balance)
        def trial_callback(study, trial):
            if trial.value is not None:
                print(f'  Trial {trial.number:>3d} | CV PR-AUC: {trial.value:.4f} | Best: {study.best_value:.4f}', flush=True)

        study.optimize(objective, n_trials=N_TRIALS, timeout=OPTUNA_TIMEOUT, callbacks=[trial_callback])

        best_params   = study.best_params
        best_cv_score = study.best_value
        n_trials_done = len(study.trials)

        print(f'  Trials selesai   : {n_trials_done}')
        print(f'  Best CV PR-AUC   : {best_cv_score:.4f}')

        # ── Refit pada seluruh X_train ────────────────────────────────────────
        if self.balance == 'smote':
            X_fit, y_fit = SMOTEDataProvider.get_full()
        else:
            X_fit, y_fit = X_train, y_train

        full_params = SearchSpaceBuilder.reconstruct(self.model_name, best_params)
        best_model = ModelFactory.build(self.model_name, self.balance, full_params)
        best_model.fit(X_fit, y_fit)

        # ── Evaluasi di val set ───────────────────────────────────────────────
        y_prob_val     = best_model.predict_proba(X_val)[:, 1]
        val_pr_auc     = average_precision_score(y_val, y_prob_val)
        delta          = val_pr_auc - self.baseline_pr_auc

        print(f'  Val PR-AUC       : {val_pr_auc:.4f}  (delta {delta:+.4f} vs baseline)')

        elapsed = time.time() - t_start
        print(f'  Waktu            : {elapsed/60:.1f} menit')

        # ── Simpan model ──────────────────────────────────────────────────────
        model_path = os.path.join(OUTPUT_DIR, f'tuned_{self.key}.joblib')
        joblib.dump(best_model, model_path)

        # ── Simpan best_params ────────────────────────────────────────────────
        params_path = os.path.join(OUTPUT_DIR, f'best_params_{self.key}.json')
        with open(params_path, 'w') as f:
            json.dump(best_params, f, indent=2)

        # ── Simpan Optuna study (untuk resume jika diperlukan) ────────────────
        study_path = os.path.join(OUTPUT_DIR, f'study_{self.key}.pkl')
        joblib.dump(study, study_path)

        # ── Log ke WandB ──────────────────────────────────────────────────────
        run = wandb.init(
            project = WANDB_PROJECT,
            group   = WANDB_TUNE_GROUP,
            name    = self.key,
            config  = {
                'model_name'      : self.model_name,
                'balance'         : self.balance,
                'n_trials'        : n_trials_done,
                'n_cv_folds'      : N_CV_FOLDS,
                **best_params,
            },
            reinit = True,
        )
        wandb.log({
            'best_cv_pr_auc'   : best_cv_score,
            'val_pr_auc'       : val_pr_auc,
            'baseline_pr_auc'  : self.baseline_pr_auc,
            'delta_vs_baseline': delta,
            'elapsed_minutes'  : elapsed / 60,
        })

        # Log trial history sebagai WandB Table
        trial_data = [
            [t.number, t.value, t.state.name]
            for t in study.trials if t.value is not None
        ]
        tbl = wandb.Table(columns=['trial', 'cv_pr_auc', 'state'], data=trial_data)
        wandb.log({'trial_history': tbl})
        wandb.finish()

        return {
            'key'             : self.key,
            'model_name'      : self.model_name,
            'balance'         : self.balance,
            'baseline_pr_auc' : self.baseline_pr_auc,
            'best_cv_pr_auc'  : best_cv_score,
            'val_pr_auc'      : val_pr_auc,
            'delta'           : delta,
            'best_params'     : best_params,
            'model_path'      : model_path,
            'elapsed_minutes' : elapsed / 60,
        }

print('OK TuningRunner terdefinisi.')

OK TuningRunner terdefinisi.


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: EnsembleWeightRunner
# Tanggung jawab: mencari bobot optimal untuk Voting Ensemble dari top-K model
# Menggunakan Optuna untuk eksplorasi kombinasi bobot integer [1,5]
# ══════════════════════════════════════════════════════════════════════════════
class EnsembleWeightRunner:
    """
    Weight search untuk Voting Ensemble (soft voting).
    Komposisi: top-3 model post-tuning berdasarkan val_pr_auc.
    Bobot dieksplorasi sebagai integer [1,5] — lebih dari 5 adalah overkill
    untuk 3 model dengan performa yang sudah cukup dekat.
    Evaluasi menggunakan val set (bukan CV) karena base models sudah di-fit.
    """

    def __init__(self, tuning_results: list):
        # Ambil top-K model berdasarkan val_pr_auc
        sorted_results = sorted(tuning_results, key=lambda x: x['val_pr_auc'], reverse=True)
        self.top_k     = sorted_results[:TOP_K_FOR_ENSEMBLE]
        print(f'  Top-{TOP_K_FOR_ENSEMBLE} model untuk ensemble:')
        for r in self.top_k:
            print(f'    {r["key"]:45s} val_PR-AUC={r["val_pr_auc"]:.4f}')

    def _load_models(self) -> list:
        """Load tuned models dari disk."""
        models = []
        for r in self.top_k:
            m = joblib.load(r['model_path'])
            safe_name = r['key'].replace('__', '_')
            models.append((safe_name, m))
        return models

    def _make_ensemble(self, weights: list, models: list) -> VotingClassifier:
        return VotingClassifier(estimators=models, voting='soft', weights=weights)

    def _objective(self, trial: optuna.Trial, models: list) -> float:
        weights = [
            trial.suggest_int(f'w_{i}', 1, 5)
            for i in range(len(models))
        ]
        ensemble  = self._make_ensemble(weights, models)
        ensemble.fit(X_train, y_train)   # base models sudah di-fit, ini hanya wrapping
        y_prob    = ensemble.predict_proba(X_val)[:, 1]
        return average_precision_score(y_val, y_prob)

    def run(self) -> dict:
        SEP = '=' * 65
        print(f'\n{SEP}')
        print(f'  Voting Ensemble — weight search ({N_TRIALS_ENSEMBLE} trials)')
        print(SEP)
        t_start = time.time()

        models = self._load_models()

        study = optuna.create_study(
            direction = 'maximize',
            sampler   = TPESampler(seed=RANDOM_SEED),
        )
        def ens_callback(study, trial):
            if trial.value is not None:
                print(f'  Trial {trial.number:>3d} | Val PR-AUC: {trial.value:.4f} | Best: {study.best_value:.4f}', flush=True)

        study.optimize(
            lambda trial: self._objective(trial, models),
            n_trials  = N_TRIALS_ENSEMBLE,
            callbacks = [ens_callback],
        )

        best_weights   = [study.best_params[f'w_{i}'] for i in range(len(models))]
        best_ensemble  = self._make_ensemble(best_weights, models)
        best_ensemble.fit(X_train, y_train)

        y_prob_val  = best_ensemble.predict_proba(X_val)[:, 1]
        val_pr_auc  = average_precision_score(y_val, y_prob_val)
        elapsed     = time.time() - t_start

        print(f'  Best weights     : {best_weights}')
        print(f'  Val PR-AUC       : {val_pr_auc:.4f}')
        print(f'  Waktu            : {elapsed/60:.1f} menit')

        # Simpan ensemble
        ens_path = os.path.join(OUTPUT_DIR, 'tuned_voting_ensemble.joblib')
        joblib.dump(best_ensemble, ens_path)

        # WandB
        component_names = [r['key'] for r in self.top_k]
        run = wandb.init(
            project = WANDB_PROJECT,
            group   = WANDB_TUNE_GROUP,
            name    = 'voting_ensemble',
            config  = {
                'components'  : component_names,
                'best_weights': best_weights,
                'n_trials'    : N_TRIALS_ENSEMBLE,
                'voting'      : 'soft',
            },
            reinit = True,
        )
        wandb.log({'val_pr_auc': val_pr_auc, 'elapsed_minutes': elapsed / 60})
        wandb.finish()

        return {
            'key'         : 'voting_ensemble',
            'components'  : component_names,
            'best_weights': best_weights,
            'val_pr_auc'  : val_pr_auc,
            'model_path'  : ens_path,
        }

print('OK EnsembleWeightRunner terdefinisi.')
print('\nOK Semua class terdefinisi. Siap eksekusi.')

OK EnsembleWeightRunner terdefinisi.

OK Semua class terdefinisi. Siap eksekusi.


---
## Main Tuning Loop

Urutan eksekusi berdasarkan estimasi waktu (tercepat lebih dulu):
1. Logistic Regression × 2 (~5 menit total)
2. XGBoost × 2 (~60–90 menit)
3. LightGBM × 2 (~90–120 menit)
4. Voting Ensemble weight search (~15 menit)

In [14]:
# ── Urutkan kandidat: LR → XGB → LGBM ────────────────────────────────────────
ORDER_MAP = {'logistic_regression': 0, 'xgboost': 1, 'lightgbm': 2}
CANDIDATES_ORDERED = sorted(CANDIDATES, key=lambda c: ORDER_MAP[c['model_name']])

tuning_results = []

for cand in CANDIDATES_ORDERED:
    runner = TuningRunner(
        model_name      = cand['model_name'],
        balance         = cand['balance'],
        baseline_pr_auc = cand['baseline_pr_auc'],
    )
    result = runner.run()
    tuning_results.append(result)

print('\n' + '='*65)
print('  TUNING INDIVIDU SELESAI')
print('='*65)


  Tuning: logistic_regression__class_weight  |  baseline val_PR-AUC=0.7410
  Trial   0 | CV PR-AUC: 0.7374 | Best: 0.7374
  Trial   1 | CV PR-AUC: 0.7363 | Best: 0.7374
  Trial   2 | CV PR-AUC: 0.7367 | Best: 0.7374
  Trial   3 | CV PR-AUC: 0.7375 | Best: 0.7375
  Trial   4 | CV PR-AUC: 0.7375 | Best: 0.7375
  Trial   5 | CV PR-AUC: 0.7374 | Best: 0.7375
  Trial   6 | CV PR-AUC: 0.7375 | Best: 0.7375
  Trial   7 | CV PR-AUC: 0.7374 | Best: 0.7375
  Trial   8 | CV PR-AUC: 0.7375 | Best: 0.7375
  Trial   9 | CV PR-AUC: 0.7374 | Best: 0.7375
  Trial  10 | CV PR-AUC: 0.7375 | Best: 0.7375
  Trial  11 | CV PR-AUC: 0.7375 | Best: 0.7375
  Trial  12 | CV PR-AUC: 0.7374 | Best: 0.7375
  Trial  13 | CV PR-AUC: 0.7375 | Best: 0.7375
  Trial  14 | CV PR-AUC: 0.7374 | Best: 0.7375
  Trial  15 | CV PR-AUC: 0.7374 | Best: 0.7375
  Trial  16 | CV PR-AUC: 0.7374 | Best: 0.7375
  Trial  17 | CV PR-AUC: 0.7375 | Best: 0.7375
  Trial  18 | CV PR-AUC: 0.7374 | Best: 0.7375
  Trial  19 | CV PR-AUC: 0.7374

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260405_060549-c09k3hij
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run logistic_regression__class_weight
wandb: ⭐️ View project at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction
wandb: 🚀 View run at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/c09k3hij
wandb: updating run metadata; uploading artifact run-c09k3hij-trial_history
wandb: uploading artifact run-c09k3hij-trial_history
wandb: 
wandb: Run history:
wandb:   baseline_pr_auc ▁
wandb:    best_cv_pr_auc ▁
wandb: delta_vs_baseline ▁
wandb:   elapsed_minutes ▁
wandb:        val_pr_auc ▁
wandb: 
wandb: Run summary:
wandb:   baseline_pr_auc 0.741
wandb:    best_cv_pr_auc 0.73754
wandb: delta_vs_baseline -0.0
wandb:   e


  Tuning: logistic_regression__smote  |  baseline val_PR-AUC=0.7407
  Trial   0 | CV PR-AUC: 0.7372 | Best: 0.7372
  Trial   1 | CV PR-AUC: 0.7369 | Best: 0.7372
  Trial   2 | CV PR-AUC: 0.7369 | Best: 0.7372
  Trial   3 | CV PR-AUC: 0.7373 | Best: 0.7373
  Trial   4 | CV PR-AUC: 0.7372 | Best: 0.7373
  Trial   5 | CV PR-AUC: 0.7372 | Best: 0.7373
  Trial   6 | CV PR-AUC: 0.7373 | Best: 0.7373
  Trial   7 | CV PR-AUC: 0.7372 | Best: 0.7373
  Trial   8 | CV PR-AUC: 0.7372 | Best: 0.7373
  Trial   9 | CV PR-AUC: 0.7372 | Best: 0.7373
  Trial  10 | CV PR-AUC: 0.7372 | Best: 0.7373
  Trial  11 | CV PR-AUC: 0.7373 | Best: 0.7373
  Trial  12 | CV PR-AUC: 0.7373 | Best: 0.7373
  Trial  13 | CV PR-AUC: 0.7373 | Best: 0.7373
  Trial  14 | CV PR-AUC: 0.7372 | Best: 0.7373
  Trial  15 | CV PR-AUC: 0.7372 | Best: 0.7373
  Trial  16 | CV PR-AUC: 0.7371 | Best: 0.7373
  Trial  17 | CV PR-AUC: 0.7372 | Best: 0.7373
  Trial  18 | CV PR-AUC: 0.7372 | Best: 0.7373
  Trial  19 | CV PR-AUC: 0.7372 | Best

wandb: setting up run np3b0ko8
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260405_075818-np3b0ko8
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run logistic_regression__smote
wandb: ⭐️ View project at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction
wandb: 🚀 View run at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/np3b0ko8
wandb: updating run metadata; uploading artifact run-np3b0ko8-trial_history
wandb: uploading artifact run-np3b0ko8-trial_history
wandb: uploading history steps 0-1, summary
wandb: 
wandb: Run history:
wandb:   baseline_pr_auc ▁
wandb:    best_cv_pr_auc ▁
wandb: delta_vs_baseline ▁
wandb:   elapsed_minutes ▁
wandb:        val_pr_auc ▁
wandb: 
wandb: Run summary:
wandb:   baseline_pr_auc 0.7407
wandb:    best_cv_pr_auc 0.73736
wandb: delta_vs_baseline -0.00014
wandb:   elapsed_minutes 112.39622
wandb:        val_p


  Tuning: xgboost__class_weight  |  baseline val_PR-AUC=0.7512
  Trial   0 | CV PR-AUC: 0.7481 | Best: 0.7481
  Trial   1 | CV PR-AUC: 0.7481 | Best: 0.7481
  Trial   2 | CV PR-AUC: 0.7470 | Best: 0.7481
  Trial   3 | CV PR-AUC: 0.7457 | Best: 0.7481
  Trial   4 | CV PR-AUC: 0.7366 | Best: 0.7481
  Trial   5 | CV PR-AUC: 0.7365 | Best: 0.7481
  Trial   6 | CV PR-AUC: 0.7492 | Best: 0.7492
  Trial   7 | CV PR-AUC: 0.7459 | Best: 0.7492
  Trial   8 | CV PR-AUC: 0.7495 | Best: 0.7495
  Trial   9 | CV PR-AUC: 0.7488 | Best: 0.7495
  Trial  10 | CV PR-AUC: 0.7493 | Best: 0.7495
  Trial  11 | CV PR-AUC: 0.7492 | Best: 0.7495
  Trial  12 | CV PR-AUC: 0.7494 | Best: 0.7495
  Trial  13 | CV PR-AUC: 0.7490 | Best: 0.7495
  Trial  14 | CV PR-AUC: 0.7495 | Best: 0.7495
  Trial  15 | CV PR-AUC: 0.7494 | Best: 0.7495
  Trial  16 | CV PR-AUC: 0.7493 | Best: 0.7495
  Trial  17 | CV PR-AUC: 0.7488 | Best: 0.7495
  Trial  18 | CV PR-AUC: 0.7493 | Best: 0.7495
  Trial  19 | CV PR-AUC: 0.7493 | Best: 0.7

...


  Tuning: lightgbm__class_weight  |  baseline val_PR-AUC=0.7514
  Trial   0 | CV PR-AUC: 0.7491 | Best: 0.7491
  Trial   1 | CV PR-AUC: 0.7486 | Best: 0.7491
  Trial   2 | CV PR-AUC: 0.7480 | Best: 0.7491
  Trial   3 | CV PR-AUC: 0.7488 | Best: 0.7491
  Trial   4 | CV PR-AUC: 0.7490 | Best: 0.7491
  Trial   5 | CV PR-AUC: 0.7492 | Best: 0.7492
  Trial   6 | CV PR-AUC: 0.7459 | Best: 0.7492
  Trial   7 | CV PR-AUC: 0.7473 | Best: 0.7492
  Trial   8 | CV PR-AUC: 0.7487 | Best: 0.7492
  Trial   9 | CV PR-AUC: 0.7470 | Best: 0.7492
  Trial  10 | CV PR-AUC: 0.7449 | Best: 0.7492
  Trial  11 | CV PR-AUC: 0.7468 | Best: 0.7492
  Trial  12 | CV PR-AUC: 0.7477 | Best: 0.7492
  Trial  13 | CV PR-AUC: 0.7484 | Best: 0.7492
  Trial  14 | CV PR-AUC: 0.7478 | Best: 0.7492
  Trial  15 | CV PR-AUC: 0.7475 | Best: 0.7492
  Trial  16 | CV PR-AUC: 0.7499 | Best: 0.7499
  Trial  17 | CV PR-AUC: 0.7494 | Best: 0.7499
  Trial  18 | CV PR-AUC: 0.7496 | Best: 0.7499
  Trial  19 | CV PR-AUC: 0.7486 | Best: 0.

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260405_121626-84xfley1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lightgbm__class_weight
wandb: ⭐️ View project at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction
wandb: 🚀 View run at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/84xfley1
wandb: updating run metadata; uploading artifact run-84xfley1-trial_history
wandb: uploading artifact run-84xfley1-trial_history
wandb: uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt; uploading media/table/trial_history_1_f14bc850bec9e7667c33.table.json; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:   baseline_pr_auc ▁
wandb:    best_cv_pr_auc ▁
wandb: delta_vs_baseline ▁
wandb:   elapsed_minutes ▁
wandb:        val_pr_auc ▁
wandb: 
wandb: Run summary:
wandb:   baseline_pr_auc 0.7514
wandb:    b


  Tuning: lightgbm__smote  |  baseline val_PR-AUC=0.7464
  Trial   0 | CV PR-AUC: 0.7434 | Best: 0.7434
  Trial   1 | CV PR-AUC: 0.7426 | Best: 0.7434
  Trial   2 | CV PR-AUC: 0.7445 | Best: 0.7445
  Trial   3 | CV PR-AUC: 0.7431 | Best: 0.7445
  Trial   4 | CV PR-AUC: 0.7441 | Best: 0.7445
  Trial   5 | CV PR-AUC: 0.7434 | Best: 0.7445
  Trial   6 | CV PR-AUC: 0.7426 | Best: 0.7445
  Trial   7 | CV PR-AUC: 0.7439 | Best: 0.7445
  Trial   8 | CV PR-AUC: 0.7442 | Best: 0.7445
  Trial   9 | CV PR-AUC: 0.7433 | Best: 0.7445
  Trial  10 | CV PR-AUC: 0.7440 | Best: 0.7445
  Trial  11 | CV PR-AUC: 0.7419 | Best: 0.7445
  Trial  12 | CV PR-AUC: 0.7442 | Best: 0.7445
  Trial  13 | CV PR-AUC: 0.7447 | Best: 0.7447
  Trial  14 | CV PR-AUC: 0.7446 | Best: 0.7447
  Trial  15 | CV PR-AUC: 0.7443 | Best: 0.7447
  Trial  16 | CV PR-AUC: 0.7445 | Best: 0.7447
  Trial  17 | CV PR-AUC: 0.7446 | Best: 0.7447
  Trial  18 | CV PR-AUC: 0.7444 | Best: 0.7447
  Trial  19 | CV PR-AUC: 0.7439 | Best: 0.7447
  

wandb: setting up run eg42h1da
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260405_142149-eg42h1da
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lightgbm__smote
wandb: ⭐️ View project at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction
wandb: 🚀 View run at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/eg42h1da
wandb: updating run metadata; uploading artifact run-eg42h1da-trial_history
wandb: uploading artifact run-eg42h1da-trial_history
wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading media/table/trial_history_1_85c3a3fcbda3925007bd.table.json; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 0-1, summary
wandb: 
wandb: Run history:
wandb:   baseline_pr_auc ▁
wandb:    best_cv_pr_auc ▁
wandb: delta_vs_baseline ▁
wandb:   elapsed_minutes ▁
wandb:        val_pr_auc ▁
wan


  TUNING INDIVIDU SELESAI


---
## Voting Ensemble — Weight Search

In [15]:
ens_runner  = EnsembleWeightRunner(tuning_results)
ens_result  = ens_runner.run()

  Top-3 model untuk ensemble:
    lightgbm__class_weight                        val_PR-AUC=0.7522
    xgboost__class_weight                         val_PR-AUC=0.7521
    xgboost__smote                                val_PR-AUC=0.7485

  Voting Ensemble — weight search (30 trials)
  Trial   0 | Val PR-AUC: 0.7523 | Best: 0.7523
  Trial   1 | Val PR-AUC: 0.7525 | Best: 0.7525
  Trial   2 | Val PR-AUC: 0.7523 | Best: 0.7525
  Trial   3 | Val PR-AUC: 0.7523 | Best: 0.7525
  Trial   4 | Val PR-AUC: 0.7525 | Best: 0.7525
  Trial   5 | Val PR-AUC: 0.7522 | Best: 0.7525
  Trial   6 | Val PR-AUC: 0.7524 | Best: 0.7525
  Trial   7 | Val PR-AUC: 0.7523 | Best: 0.7525
  Trial   8 | Val PR-AUC: 0.7525 | Best: 0.7525
  Trial   9 | Val PR-AUC: 0.7525 | Best: 0.7525
  Trial  10 | Val PR-AUC: 0.7525 | Best: 0.7525
  Trial  11 | Val PR-AUC: 0.7525 | Best: 0.7525
  Trial  12 | Val PR-AUC: 0.7525 | Best: 0.7525
  Trial  13 | Val PR-AUC: 0.7525 | Best: 0.7525
  Trial  14 | Val PR-AUC: 0.7525 | Best: 0.7525

wandb: setting up run ircshkvg
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260405_145216-ircshkvg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run voting_ensemble
wandb: ⭐️ View project at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction
wandb: 🚀 View run at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/ircshkvg
wandb: updating run metadata; uploading summary
wandb: uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt
wandb: 
wandb: Run history:
wandb: elapsed_minutes ▁
wandb:      val_pr_auc ▁
wandb: 
wandb: Run summary:
wandb: elapsed_minutes 30.38906
wandb:      val_pr_auc 0.75254
wandb: 
wandb: 🚀 View run voting_ensemble at: https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/ircshkvg
wandb: ⭐️ View project at: https://wandb.ai/ardiyan

---
## Ringkasan Hasil Tuning

In [16]:
print('=== RINGKASAN TUNING NOTEBOOK 05 ===\n')

header = f'  {"Model":<45} {"Baseline":>10} {"Tuned":>10} {"Delta":>8} {"Waktu":>8}'
sep_line = f'  {"-"*45} {"-"*10} {"-"*10} {"-"*8} {"-"*8}'
print(header)
print(sep_line)

for r in sorted(tuning_results, key=lambda x: x['val_pr_auc'], reverse=True):
    delta_str = f"{r['delta']:+.4f}"
    print(
        f'  {r["key"]:<45}'
        f' {r["baseline_pr_auc"]:>10.4f}'
        f' {r["val_pr_auc"]:>10.4f}'
        f' {delta_str:>8}'
        f' {r["elapsed_minutes"]:>6.1f}m'
    )

print(f'\n  {"voting_ensemble":<45}', end='')
print(f' {"":>10} {ens_result["val_pr_auc"]:>10.4f}')
print(f'\n  Components  : {ens_result["components"]}')
print(f'  Weights     : {ens_result["best_weights"]}')

=== RINGKASAN TUNING NOTEBOOK 05 ===

  Model                                           Baseline      Tuned    Delta    Waktu
  --------------------------------------------- ---------- ---------- -------- --------
  lightgbm__class_weight                            0.7514     0.7522  +0.0008   60.4m
  xgboost__class_weight                             0.7512     0.7521  +0.0009   61.9m
  xgboost__smote                                    0.7479     0.7485  +0.0006  135.7m
  lightgbm__smote                                   0.7464     0.7477  +0.0013  125.3m
  logistic_regression__class_weight                 0.7410     0.7410  -0.0000   21.0m
  logistic_regression__smote                        0.7407     0.7406  -0.0001  112.4m

  voting_ensemble                                              0.7525

  Components  : ['lightgbm__class_weight', 'xgboost__class_weight', 'xgboost__smote']
  Weights     : [5, 3, 1]


In [17]:
# ── Identifikasi model terbaik (untuk XAI Gate #2) ───────────────────────────
all_results = tuning_results + [ens_result]

best_individual = max(tuning_results, key=lambda x: x['val_pr_auc'])
best_overall    = max(all_results,    key=lambda x: x['val_pr_auc'])

print(f'  Best individual : {best_individual["key"]}  PR-AUC={best_individual["val_pr_auc"]:.4f}')
print(f'  Best overall    : {best_overall["key"]}  PR-AUC={best_overall["val_pr_auc"]:.4f}')

# Cek apakah ada model yang regress dari baseline
print('\n  Regresi vs baseline:')
for r in tuning_results:
    if r['delta'] < 0:
        print(f'    PERHATIAN: {r["key"]} turun {r["delta"]:+.4f}')
    else:
        print(f'    OK: {r["key"]} naik {r["delta"]:+.4f}')

  Best individual : lightgbm__class_weight  PR-AUC=0.7522
  Best overall    : voting_ensemble  PR-AUC=0.7525

  Regresi vs baseline:
    PERHATIAN: logistic_regression__class_weight turun -0.0000
    PERHATIAN: logistic_regression__smote turun -0.0001
    OK: xgboost__class_weight naik +0.0009
    OK: xgboost__smote naik +0.0006
    OK: lightgbm__class_weight naik +0.0008
    OK: lightgbm__smote naik +0.0013


---
## Visualisasi Perbandingan

In [18]:
fig, ax = plt.subplots(figsize=(12, 5))

keys      = [r['key'] for r in tuning_results]
baselines = [r['baseline_pr_auc'] for r in tuning_results]
tuned     = [r['val_pr_auc']      for r in tuning_results]

x      = np.arange(len(keys))
width  = 0.35

bars_base  = ax.bar(x - width/2, baselines, width, label='Baseline', color='#B4B2A9', alpha=0.85)
bars_tuned = ax.bar(x + width/2, tuned,     width, label='Tuned',    color='#1D9E75', alpha=0.85)

# Delta label di atas bar tuned
for bar, r in zip(bars_tuned, tuning_results):
    delta  = r['delta']
    color  = '#0F6E56' if delta >= 0 else '#993C1D'
    label  = f'{delta:+.4f}'
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.001,
        label, ha='center', va='bottom',
        fontsize=8, color=color, fontweight='bold'
    )

# Ensemble line
ax.axhline(
    ens_result['val_pr_auc'], color='#534AB7',
    linestyle='--', linewidth=1.5, label=f'Voting Ensemble ({ens_result["val_pr_auc"]:.4f})'
)

ax.set_xlabel('Model', fontsize=11)
ax.set_ylabel('val PR-AUC', fontsize=11)
ax.set_title('Baseline vs Tuned — val PR-AUC per Kandidat', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(keys, rotation=15, ha='right', fontsize=9)
ax.legend(fontsize=9)
ax.set_ylim(0.70, min(1.0, max(tuned + [ens_result['val_pr_auc']]) + 0.02))
ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.4f'))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'tuning_comparison.png')
plt.savefig(plot_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Plot disimpan: {plot_path}')

Plot disimpan: /kaggle/working/artifacts/tuning_comparison.png


---
## Save Output untuk Notebook 06 (XAI Gate #2)

In [19]:
def make_serializable(obj):
    """Konversi numpy types ke Python native untuk JSON serialization."""
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    if isinstance(obj, dict):           return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):           return [make_serializable(i) for i in obj]
    return obj

summary = {
    'notebook'         : 'Notebook 05 — Hyperparameter Tuning',
    'n_cv_folds'       : N_CV_FOLDS,
    'n_trials_per_model': N_TRIALS,
    'n_trials_ensemble': N_TRIALS_ENSEMBLE,
    'individual_results': make_serializable(tuning_results),
    'ensemble_result'  : make_serializable(ens_result),
    'best_individual'  : best_individual['key'],
    'best_overall'     : best_overall['key'],
}

summary_path = os.path.join(OUTPUT_DIR, 'tuning_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'OK tuning_summary.json disimpan: {summary_path}')
print(json.dumps({
    'total_candidates'  : len(tuning_results),
    'best_individual'   : best_individual['key'],
    'best_individual_pr': round(best_individual['val_pr_auc'], 4),
    'ensemble_pr'       : round(ens_result['val_pr_auc'], 4),
}, indent=2))

OK tuning_summary.json disimpan: /kaggle/working/artifacts/tuning_summary.json
{
  "total_candidates": 6,
  "best_individual": "lightgbm__class_weight",
  "best_individual_pr": 0.7522,
  "ensemble_pr": 0.7525
}


---
## Ringkasan Final

In [20]:
import glob

SEP = '=' * 65
print(SEP)
print('  NOTEBOOK 05 SELESAI')
print(SEP)
print()
print(f'  Total kandidat individual : {len(tuning_results)}')
print(f'  CV strategy              : {N_CV_FOLDS}-fold Stratified CV')
print(f'  Trials per model         : {N_TRIALS}')
print(f'  Trials ensemble          : {N_TRIALS_ENSEMBLE}')
print()
print('  Hasil akhir (diurutkan val_PR-AUC):')
for r in sorted(tuning_results, key=lambda x: x['val_pr_auc'], reverse=True):
    flag = '*' if r['key'] == best_individual['key'] else ' '
    print(f'  {flag} {r["key"]:<45} val_PR-AUC={r["val_pr_auc"]:.4f}  d{r["delta"]:+.4f}')
print(f'    {"voting_ensemble":<46} val_PR-AUC={ens_result["val_pr_auc"]:.4f}')

print()
print('  Artifacts:')
for fpath in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*'))):
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {os.path.basename(fpath):<55} ({size_kb:.0f} KB)')

print()
print('  Langkah berikutnya: 06_xai_gate2/')
print('    Evaluasi D1-D4 post-tuning untuk semua kandidat')
print('    SHAP penuh untuk Voting Ensemble')

  NOTEBOOK 05 SELESAI

  Total kandidat individual : 6
  CV strategy              : 3-fold Stratified CV
  Trials per model         : 100
  Trials ensemble          : 30

  Hasil akhir (diurutkan val_PR-AUC):
  * lightgbm__class_weight                        val_PR-AUC=0.7522  d+0.0008
    xgboost__class_weight                         val_PR-AUC=0.7521  d+0.0009
    xgboost__smote                                val_PR-AUC=0.7485  d+0.0006
    lightgbm__smote                               val_PR-AUC=0.7477  d+0.0013
    logistic_regression__class_weight             val_PR-AUC=0.7410  d-0.0000
    logistic_regression__smote                    val_PR-AUC=0.7406  d-0.0001
    voting_ensemble                                val_PR-AUC=0.7525

  Artifacts:
  best_params_lightgbm__class_weight.json                 (0 KB)
  best_params_lightgbm__smote.json                        (0 KB)
  best_params_logistic_regression__class_weight.json      (0 KB)
  best_params_logistic_regression__smote.json